# Day 16: Predict the Predictor — Retail AI Behavior and Information Asymmetry in the AI Era

*Alpha Flow Research · HongJin HE · July 2026*

---

## A New Regime: The AI-Augmented Retail Investor

Until 2022, retail investors were characterized by:
- High emotional variance (fear/greed cycles)
- Low information quality (noise traders)
- Unpredictable, idiosyncratic behavior

After 2023 (ChatGPT mass adoption), a structural shift occurred:
- Retail investors now query LLMs for investment advice at scale
- Their decisions are increasingly **conditioned on AI outputs**
- AI outputs are **deterministic given the prompt + model version**
- Their behavior becomes **more predictable**, not less

> *Paradox: By using AI to become "more rational", retail investors become more predictable to sophisticated counterparties.*

## The Arbitrage: Predict the AI's Prediction

If we know:
1. What information retail investors have access to (public data, news)
2. What questions they are likely to ask (from social media patterns)
3. What the AI answers (we can query the same models)

Then we can estimate the **retail strategy distribution** $\mu^{\text{retail}}_t$ *before the retail investors act*.

This is the most novel application of MicroWorld's MFG framework:
$$\mu^{\text{retail}}_t = \text{LLM}(\text{PublicInfo}_t, \text{RetailQuery}_t) \to \text{Nash adjustment}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List
from enum import Enum

np.random.seed(2026)

# ── Model: AI-era information asymmetry ──────────────────────────────────

class InvestorType(Enum):
    RETAIL_PRE_AI   = 'retail_pre_ai'    # emotional, unpredictable
    RETAIL_AI_AIDED = 'retail_ai_aided'  # AI-augmented, more predictable
    INSTITUTIONAL   = 'institutional'    # clean data, proprietary models
    QUANT_FIRM      = 'quant_firm'       # systematic, alpha-seeking

@dataclass
class InvestorProfile:
    investor_type: InvestorType
    population_share: float   # fraction of market by count
    capital_share: float      # fraction of AUM
    signal_quality: float     # 0=pure noise, 1=perfect signal
    decision_lag_days: float  # how long to act after signal
    behavior_variance: float  # idiosyncratic noise in decisions
    ai_adoption_rate: float   # fraction using AI (0-1)

# Market composition (approximate 2026 US equity market)
investors = [
    InvestorProfile(InvestorType.RETAIL_PRE_AI,   0.30, 0.05, 0.10, 3.0, 0.8, 0.0),
    InvestorProfile(InvestorType.RETAIL_AI_AIDED, 0.35, 0.07, 0.35, 1.0, 0.3, 1.0),
    InvestorProfile(InvestorType.INSTITUTIONAL,   0.25, 0.55, 0.75, 0.5, 0.1, 0.3),
    InvestorProfile(InvestorType.QUANT_FIRM,      0.10, 0.33, 0.90, 0.0, 0.05, 0.5),
]

# Show how AI adoption changes predictability landscape
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

names = [i.investor_type.value.replace('_', '\n') for i in investors]
colors = ['coral', 'orange', 'steelblue', 'green']

# Plot 1: Population share
axes[0].bar(names, [i.population_share for i in investors], color=colors, alpha=0.8)
axes[0].set_title('Population Share')
axes[0].set_ylabel('Fraction of participants')

# Plot 2: Signal quality vs behavior variance (predictability map)
for inv, color in zip(investors, colors):
    axes[1].scatter(inv.signal_quality, 1 - inv.behavior_variance,
                   s=inv.capital_share * 1000, color=color, alpha=0.8,
                   label=inv.investor_type.value.replace('_', ' '),
                   edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Signal quality (0=noise, 1=perfect)')
axes[1].set_ylabel('Predictability (1=fully predictable)')
axes[1].set_title('Who Is Most Predictable?\n(bubble size = AUM share)')
axes[1].legend(fontsize=8)
axes[1].axvline(0.5, color='gray', ls='--', alpha=0.5)
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.5)

# Plot 3: Information convergence with AI adoption
ai_adoption = np.linspace(0, 1, 100)
# Signal quality of retail improves with AI, but asymptotes below institutional
retail_signal = 0.10 + 0.35 * ai_adoption / (1 + 0.5 * ai_adoption)  # diminishing returns
institutional_signal = np.full(100, 0.75)
quant_signal = np.full(100, 0.90)
gap = quant_signal - retail_signal
convergence_floor = 0.90 - 0.35  # irreducible gap

axes[2].plot(ai_adoption, retail_signal, color='orange', lw=2, label='AI-augmented retail')
axes[2].plot(ai_adoption, institutional_signal, color='steelblue', lw=2, ls='--', label='Institutional')
axes[2].plot(ai_adoption, quant_signal, color='green', lw=2, ls='--', label='Quant firm')
axes[2].fill_between(ai_adoption, retail_signal, quant_signal, alpha=0.2, color='red', label='Persistent gap')
axes[2].axhline(convergence_floor + 0.10, color='red', ls=':', alpha=0.7)
axes[2].set_xlabel('Retail AI adoption rate')
axes[2].set_ylabel('Signal quality')
axes[2].set_title('Information Gap Does Not Close to Zero\n(structural irreducible floor)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/ai_era_information_asymmetry.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key insight: as retail AI adoption → 1, their signal quality plateaus.')
print('Reasons for irreducible gap:')
print('  1. Data cleaning: institutions have proprietary clean data; retail has noisy public data')
print('  2. Decision speed: large institutions act faster (decision lag < retail)')
print('  3. AI quality: institutions run proprietary models; retail uses consumer LLMs')
print('  4. Capital scale: size itself creates information (prime brokerage, order flow)')

## Reverse-Engineering Retail AI Behavior

The **predict the predictor** pipeline:

```
Step 1: Observe public information at time t
        (news headlines, price moves, earnings surprises)

Step 2: Infer retail query distribution
        P(query | public_info_t)  ← from social media mining
        e.g., if NVDA drops 5%, retail queries spike: "should I buy NVDA dip?"

Step 3: Simulate AI responses
        response = LLM(query)  ← query same models retail uses
        Extract: [sentiment, action_recommendation, confidence]

Step 4: Aggregate into retail strategy distribution
        μ^retail_t = aggregate(responses × population_weights)
        This is the mean-field of the retail population

Step 5: Feed into MFG
        Use μ^retail_t as a known (not solved-for) component of μ_t
        → Reduces complexity of Nash equilibrium problem
        → Improves prediction of L2 (quant firm) optimal strategies
```

This is a **partial Nash equilibrium**: retail is modeled as a follower (their strategy is predicted from AI outputs), and quant firms solve a reduced-form game against the residual uncertainty.

In [ ]:
# Simulate the predict-the-predictor pipeline (stub — real version uses LLM API)

@dataclass
class MarketEvent:
    ticker: str
    event_type: str  # 'price_drop', 'earnings_beat', 'earnings_miss', 'news'
    magnitude: float  # percentage change
    sentiment: float  # -1 to +1

def simulate_retail_query_distribution(event: MarketEvent) -> Dict[str, float]:
    """
    Estimate distribution of retail queries given a market event.
    In production: mine Reddit/Twitter/StockTwits for actual query patterns.
    Here: rule-based stub calibrated to observed social media behavior.
    """
    queries = {}
    if event.event_type == 'price_drop':
        queries[f'Is {event.ticker} a buy the dip opportunity?'] = 0.35
        queries[f'Why is {event.ticker} dropping? Should I sell?'] = 0.30
        queries[f'What is the fair value of {event.ticker}?'] = 0.20
        queries[f'Is {event.ticker} going bankrupt?'] = 0.15
    elif event.event_type == 'earnings_beat':
        queries[f'Will {event.ticker} continue to rise after earnings?'] = 0.40
        queries[f'Is it too late to buy {event.ticker}?'] = 0.35
        queries[f'What is {event.ticker} price target?'] = 0.25
    elif event.event_type == 'earnings_miss':
        queries[f'Should I sell {event.ticker} after earnings miss?'] = 0.45
        queries[f'Will {event.ticker} recover from earnings miss?'] = 0.35
        queries[f'Compare {event.ticker} to competitors'] = 0.20
    return queries

def stub_llm_response(query: str, event: MarketEvent) -> Dict[str, float]:
    """
    Stub LLM response. In production: query GPT-4/Claude/Gemini API.
    Returns: {action: probability} distribution over Buy/Hold/Sell.
    Standard LLM investment advice tends toward: hold/DCA for drops, caution for spikes.
    """
    if 'buy the dip' in query.lower() or 'continue to rise' in query.lower():
        return {'Buy': 0.50, 'Hold': 0.35, 'Sell': 0.15}
    elif 'sell' in query.lower() or 'bankrupt' in query.lower():
        return {'Buy': 0.15, 'Hold': 0.45, 'Sell': 0.40}
    elif 'too late' in query.lower():
        return {'Buy': 0.30, 'Hold': 0.50, 'Sell': 0.20}
    else:
        return {'Buy': 0.33, 'Hold': 0.34, 'Sell': 0.33}

def aggregate_retail_strategy(event: MarketEvent, ai_adoption: float = 0.60) -> Dict:
    """Aggregate retail strategy distribution μ^retail from AI responses."""
    queries = simulate_retail_query_distribution(event)
    
    # Weighted average of AI responses
    agg = {'Buy': 0.0, 'Hold': 0.0, 'Sell': 0.0}
    for query, weight in queries.items():
        response = stub_llm_response(query, event)
        for action, prob in response.items():
            agg[action] += weight * prob
    
    # Mix with pre-AI retail behavior (emotional baseline)
    emotional_baseline = {
        'price_drop': {'Buy': 0.20, 'Hold': 0.30, 'Sell': 0.50},  # panic sell
        'earnings_beat': {'Buy': 0.60, 'Hold': 0.25, 'Sell': 0.15},  # FOMO buy
        'earnings_miss': {'Buy': 0.15, 'Hold': 0.25, 'Sell': 0.60},  # panic sell
    }.get(event.event_type, {'Buy': 0.33, 'Hold': 0.34, 'Sell': 0.33})
    
    mixed = {k: ai_adoption * agg[k] + (1 - ai_adoption) * emotional_baseline[k]
             for k in agg}
    return {'strategy_distribution': mixed, 'ai_agg': agg, 'emotional': emotional_baseline}

# Test on three market scenarios
events = [
    MarketEvent('NVDA', 'price_drop', -8.5, -0.7),
    MarketEvent('AAPL', 'earnings_beat', +6.2, +0.8),
    MarketEvent('META', 'earnings_miss', -12.3, -0.9),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, event in zip(axes, events):
    result = aggregate_retail_strategy(event, ai_adoption=0.60)
    actions = ['Buy', 'Hold', 'Sell']
    colors_bar = ['green', 'gray', 'red']
    
    x = np.arange(3)
    width = 0.25
    ax.bar(x - width, [result['emotional'][a] for a in actions], width, 
           label='Pre-AI retail', color=colors_bar, alpha=0.4)
    ax.bar(x, [result['ai_agg'][a] for a in actions], width,
           label='AI-guided', color=colors_bar, alpha=0.7)
    ax.bar(x + width, [result['strategy_distribution'][a] for a in actions], width,
           label='Mixed (60% AI)', color=colors_bar, alpha=1.0, edgecolor='black', lw=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(actions)
    ax.set_title(f'{event.ticker}: {event.event_type}\n({event.magnitude:+.1f}%)')
    ax.set_ylabel('Strategy probability')
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('Retail Strategy Distribution μ^retail: AI-Era vs Pre-AI', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/predict_the_predictor.png', dpi=150, bbox_inches='tight')
plt.show()

print('AI adoption makes retail behavior more predictable:')
for event in events:
    result = aggregate_retail_strategy(event, ai_adoption=0.60)
    pre_ai = result['emotional']
    post_ai = result['strategy_distribution']
    entropy_pre = -sum(p * np.log(p+1e-10) for p in pre_ai.values())
    entropy_post = -sum(p * np.log(p+1e-10) for p in post_ai.values())
    print(f'  {event.ticker} ({event.event_type}): entropy {entropy_pre:.3f} → {entropy_post:.3f} (lower = more predictable)')

## Integration with the MFG Nash Equilibrium

The retail strategy distribution $\mu^{\text{retail}}_t$ enters the full MFG as a **partially known component** of the mean-field:

$$\mu_t = \underbrace{\mu^{\text{retail}}_t}_{\text{known from AI prediction}} + \underbrace{\mu^{\text{inst}}_t}_{\text{Nash equilibrium}} + \underbrace{\mu^{\text{quant}}_t}_{\text{Nash equilibrium}}$$

By treating $\mu^{\text{retail}}_t$ as observable (not a strategic variable), the Nash equilibrium problem reduces from a 3-population game to a **2-population game** (institutional + quant), which is:
- Smaller → faster to solve
- Better identified → more stable Nash
- Directly actionable → Controller C can exploit the retail predictability

## Why This Is Not Manipulation

Predicting what publicly available AI tools will tell retail investors is no different from:
- Predicting what analyst reports will say (sell-side research prediction)
- Predicting how passive rebalancing will affect prices (index arbitrage)
- Predicting earnings reaction using historical patterns

All of these are **information-based strategies**, not manipulation. The retail investors could equally well try to predict what quant firms will do.

## Implementation Note

See `agents/retail_ai.py` for the full implementation. The key function `predict_retail_strategy(market_event, model='gpt-4o')` accepts a real LLM API key or runs in stub mode (no API cost) for development.